In [ ]:
import fitz # PyMuPDF

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def show(img, title="Immagine"):
    plt.figure(figsize=(10,10))
    plt.imshow(img, cmap='gray')
    plt.title(title)
    plt.axis('off')
    plt.show()


In [ ]:
%matplotlib

# Algoritmi di Interpolazione

In [ ]:
coord_riquadro_100dpi = (222, 62, 275, 81)
dpi_orig = 300
filename_immagine = f"../Lezione2/out/Lanterna-{dpi_orig}dpi_ruotata_tagliata.png"

# Carica un piccolo pezzo di testo (es. una parola)
img = cv2.imread(filename_immagine, 0)
coord_riquadro = tuple(int(c * dpi_orig / 100) for c in coord_riquadro_100dpi)
print("Coordinate riquadro:", dpi_orig, coord_riquadro)
# crop image
img = img[coord_riquadro[1]:coord_riquadro[3], coord_riquadro[0]:coord_riquadro[2]]

# Riduciamo di 3 volte con 4 metodi diversi
near_div3 = cv2.resize(img, None, fx=0.3, fy=0.3, interpolation=cv2.INTER_NEAREST)
linear_div3 = cv2.resize(img, None, fx=0.3, fy=0.3, interpolation=cv2.INTER_LINEAR)
cubic_div3 = cv2.resize(img, None, fx=0.3, fy=0.3, interpolation=cv2.INTER_CUBIC)
lanczos_div3 = cv2.resize(img, None, fx=0.3, fy=0.3, interpolation=cv2.INTER_LANCZOS4)

# Ingrandiamo di 5 volte con 4 metodi diversi
near_mult5 = cv2.resize(img, None, fx=5, fy=5, interpolation=cv2.INTER_NEAREST)
linear_mult5 = cv2.resize(img, None, fx=5, fy=5, interpolation=cv2.INTER_LINEAR)
cubic_mult5 = cv2.resize(img, None, fx=5, fy=5, interpolation=cv2.INTER_CUBIC)
lanczos_mult5 = cv2.resize(img, None, fx=5, fy=5, interpolation=cv2.INTER_LANCZOS4)

# Visualizzazione
titles = ['Originale', 'Nearest', 'Bilinear', 'Bicubic', 'Lanczos']
titles += ['Originale', 'Nearest', 'Bilinear', 'Bicubic', 'Lanczos']
images = [img, near_div3, linear_div3, cubic_div3, lanczos_div3, img, near_mult5, linear_mult5, cubic_mult5, lanczos_mult5]

for i, image in enumerate(images):
    plt.subplot(2, 5, i+1)
    plt.imshow(image, cmap='gray')
    plt.title(titles[i], fontsize=8)
    plt.axis('off')
plt.show()


# Median Blur

In [ ]:
img = cv2.imread('../Lezione2/out/Lanterna-300dpi_zoom_4_grigia_scalata.png', 0)

In [ ]:
# Rimuove i puntini mantenendo i bordi
#     del testo abbastanza nitidi
show(img, "Originale")
denoised = cv2.medianBlur(img, 11) 
show(denoised, "Median Blur")


In [ ]:

img = cv2.imread('../Lezione2/out/Lanterna-300dpi_zoom_4_grigia_scalata.png', 0)

img = img[:500,:1000]
show(img, "Originale")
# Algoritmo NLMeans
# Parametri da testare: h, templateWindowSize, searchWindowSize
denoised = cv2.fastNlMeansDenoising(img, None, h=3.0, templateWindowSize=7, searchWindowSize=21)

show(denoised, "Risultato NLMeans")

In [ ]:
# 1. Creiamo un "modello" dello sfondo usando la Dilatazione
# L'idea è far sparire il testo lasciando solo la carta sporca
kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (17,17))
background = cv2.morphologyEx(img, cv2.MORPH_DILATE, kernel)
show(background, "Sfondo")

# 2. Sottraiamo lo sfondo dall'immagine originale
# Questo isola il testo (nero) dallo sfondo (bianco)
out_gray = cv2.divide(img, background, scale=255)

show(out_gray, "Testo isolato dallo sfondo")

# CLAHE

In [ ]:
# Creazione dell'oggetto CLAHE
show(img, "Originale")
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(16,16))
cl1 = clahe.apply(img)

show(cl1, "Contrasto Adattivo CLAHE")


# Dilatazione e Erosione

In [ ]:
coord_riquadro_100dpi = (222, 62, 275, 81)

dpi_orig = 300

filename_immagine = f"../Lezione2/out/Lanterna-{dpi_orig}dpi_ruotata_tagliata.png"

# Carica un piccolo pezzo di testo (es. una parola)
img = cv2.imread(filename_immagine, 0)
coord_riquadro = tuple(int(c * dpi_orig / 100) for c in coord_riquadro_100dpi)
print("Coordinate riquadro:", dpi_orig, coord_riquadro)
# crop image
img = img[coord_riquadro[1]:coord_riquadro[3], coord_riquadro[0]:coord_riquadro[2]]

# Otsu's Thresholding
soglia, filtrata_otsu = cv2.threshold(img, 0, 255,
                                        cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# Adaptive Thresholding
#    (Ideale per documenti vecchi/ombreggiati)
filtrata_adattata = cv2.adaptiveThreshold(img, 255,
                                    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                    cv2.THRESH_BINARY, 11, 2)

kernel = np.ones((2, 2), np.uint8)

titles = ['Originale', 'Otsu', 'Adaptive'] * 3
images = [img, filtrata_otsu, filtrata_adattata]

for fun in cv2.erode, cv2.dilate:
    for filtrata in [img, filtrata_otsu, filtrata_adattata]:
        images.append(fun(filtrata, kernel, iterations=2))

for i, image in enumerate(images):
    plt.subplot(3, 3, i+1)
    plt.imshow(image, cmap='gray')
    plt.title(titles[i], fontsize=8)
    plt.axis('off')
plt.show()


# Eliminare i colori

In [ ]:
img_ruotata_tagliata = cv2.imread('../Lezione2/out/Lanterna-300dpi_ruotata_tagliata.png')

plt.imshow(img_ruotata_tagliata)
plt.title("Originale")
plt.show()

# Seleziona un'area senza testo
roi_carta = img_ruotata_tagliata[200:250, 900:950] 
show(roi_carta)

# Analisi istogramma del ritaglio
hsv_roi_carta = cv2.cvtColor(roi_carta, cv2.COLOR_BGR2HSV)
hist_carta = cv2.calcHist([hsv_roi_carta], [0, 1], None, [180, 256], [0, 180, 0, 256])

plt.imshow(hist_carta, interpolation='nearest', origin='lower', extent=[0, 256, 0, 180])
plt.xlabel('Saturation')
plt.ylabel('Hue')
plt.title('Mappa dei colori presenti nella carta')
plt.show()

lower1 = np.array([0, 20, 0])
upper1 = np.array([35, 70, 255])
mask_colore1 = cv2.inRange(cv2.cvtColor(img_ruotata_tagliata, cv2.COLOR_BGR2HSV), lower1, upper1)

lower2 = np.array([168, 18, 0])
upper2 = np.array([172, 32, 255])
mask_colore2 = cv2.inRange(cv2.cvtColor(img_ruotata_tagliata, cv2.COLOR_BGR2HSV), lower2, upper2)

mask_finale = cv2.bitwise_or(mask_colore1, mask_colore2)

img_pulita = img_ruotata_tagliata.copy()
img_pulita[mask_finale > 0] = [255, 255, 255] # Trasforma carta e macchie in bianco
plt.imshow(img_pulita)
plt.title("Pulita")
plt.show()

In [ ]:

def plot_color_histogram_3d(image_path):
    # Load the image
    img = cv2.imread(image_path)
    if img is None:
        print("Error: Image not found.")
        return
    
    # Convert from BGR (OpenCV) to RGB
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Calculate 3D histogram (using 16 bins per channel for visibility)
    # Range is 0 to 256 for Red, Green, and Blue channels
    hist, edges = np.histogramdd(
        img_rgb.reshape(-1, 3), 
        bins=(16, 16, 16), 
        range=((0, 256), (0, 256), (0, 256))
    )
    
    # Setup the 3D figure
    fig = plt.figure(figsize=(10, 7))
    ax = fig.add_subplot(111, projection='3d')
    
    # Generate coordinates for the bars
    r_edges, g_edges, b_edges = edges
    r, g, b = np.meshgrid(
        r_edges[:-1], g_edges[:-1], b_edges[:-1], indexing="ij"
    )
    
    # Flatten arrays to plot individual bars
    r_flat = r.ravel()
    g_flat = g.ravel()
    b_flat = b.ravel()
    freq = hist.ravel()
    
    # Filter out empty bins (frequency > 0) to avoid rendering hidden bars
    mask = freq > 0
    r_flat = r_flat[mask]
    g_flat = g_flat[mask]
    b_flat = b_flat[mask]
    freq = freq[mask]
    
    # Color the 3D bars based on their actual RGB value
    # Normalized by 256 so matplotlib understands the color values
    bar_colors = np.stack([r_flat / 255.0, g_flat / 255.0, b_flat / 255.0], axis=1)
    
    # Plot the 3D histogram
    ax.bar3d(r_flat, g_flat, b_flat, dx=16, dy=16, dz=freq, color=bar_colors, alpha=0.8)
    
    ax.set_xlabel('Red')
    ax.set_ylabel('Green')
    ax.set_zlabel('Frequency')
    ax.set_title('3D Color Histogram')
    
    plt.show()

# Example usage:
plot_color_histogram_3d('../Lezione2/out/Lanterna-300dpi_ruotata_tagliata.png')

# Grafici di frequenza colori

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.mplot3d import Axes3D

# 1. Sample data: R, G, B coordinates and their frequencies
r_coords = [255, 0, 128, 200]
g_coords = [0, 255, 0, 50]
b_coords = [0, 0, 128, 200]
frequencies = [15, 45, 8, 30] # Frequency of each color

# Normalize colors to [0, 1] range for matplotlib
color_normalized = np.array([r_coords, g_coords, b_coords]).T / 255.0

# Scale frequencies for marker size (s)
# formula: s = scale_factor * frequency
sizes = [50 * freq for freq in frequencies]

# 2. Initialize the 3D plot
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')

# 3. Create the 3D scatter plot
sc = ax.scatter(r_coords, g_coords, b_coords, 
                s=sizes, 
                c=color_normalized, 
                alpha=0.6, 
                edgecolors='w')

# 4. Label the axes
ax.set_xlabel('Red (R)')
ax.set_ylabel('Green (G)')
ax.set_zlabel('Blue (B)')
ax.set_title('3D Color Frequency Scatter Plot')

# Set axis ranges since RGB values are typically 0-255
ax.set_xlim([0, 255])
ax.set_ylim([0, 255])
ax.set_zlim([0, 255])

plt.show()


## Funzioni usate in seguito

In [ ]:
"""
Grafico 3D della frequenza dei colori in un'immagine.
Utilizzo: python color_frequency_3d.py [percorso_immagine]

Dipendenze:
    pip install opencv-python matplotlib numpy
"""

import sys
import numpy as np
import cv2
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.colors as mcolors


def load_image(path: str) -> np.ndarray:
    """Carica un'immagine e la converte da BGR a RGB."""
    img_bgr = cv2.imread(path)
    if img_bgr is None:
        raise FileNotFoundError(f"Impossibile aprire l'immagine: {path}")
    return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)


def quantize_colors(image: np.ndarray, bins: int = 16) -> tuple:
    """
    Riduce i valori RGB in bucket (bin) per renderli aggregabili.

    Returns:
        centers  – array (N, 3) con il centro RGB di ogni bin occupato
        counts   – array (N,)   con la frequenza di quel bin
        colors   – array (N, 4) con i colori RGBA normalizzati per matplotlib
    """
    # Reshape a lista di pixel (H*W, 3)
    pixels = image.reshape(-1, 3).astype(np.float32)

    # Calcola l'istogramma 3D nello spazio RGB
    hist, edges = np.histogramdd(
        pixels,
        bins=bins,
        range=[(0, 256), (0, 256), (0, 256)],
    )

    # Centri dei bin
    cx = (edges[0][:-1] + edges[0][1:]) / 2
    cy = (edges[1][:-1] + edges[1][1:]) / 2
    cz = (edges[2][:-1] + edges[2][1:]) / 2

    # Indici dei bin non vuoti
    r_idx, g_idx, b_idx = np.where(hist > 0)
    counts = hist[r_idx, g_idx, b_idx]

    centers = np.stack([cx[r_idx], cy[g_idx], cz[b_idx]], axis=1)

    # Colori RGBA per ogni punto (normalizzati in [0,1])
    colors = np.column_stack([centers / 255.0, np.ones(len(centers))])

    return centers, counts, colors


def plot_3d_color_frequency(
    image: np.ndarray,
    bins: int = 16,
    top_n: int | None = None,
    title: str = "Frequenza dei colori 3D",
) -> None:
    """
    Visualizza la frequenza dei colori come scatter plot 3D nello spazio RGB.

    Args:
        image  – array NumPy (H, W, 3) RGB
        bins   – numero di bucket per canale (più basso = più aggregazione)
        top_n  – se specificato, mostra solo i top_n colori più frequenti
        title  – titolo del grafico
    """
    centers, counts, colors = quantize_colors(image, bins=bins)

    # Filtra ai top N se richiesto
    if top_n is not None:
        idx = np.argsort(counts)[::-1][:top_n]
        centers, counts, colors = centers[idx], counts[idx], colors[idx]

    # Normalizza i counts per la dimensione dei marker
    size_norm = (counts / counts.max()) * 300 + 10

    fig = plt.figure(figsize=(14, 7))
    fig.patch.set_facecolor("#0e0e12")

    # ── Pannello sinistro: scatter 3D ────────────────────────────────────────
    ax3d = fig.add_subplot(121, projection="3d")
    ax3d.set_facecolor("#fefef2")

    sc = ax3d.scatter(
        centers[:, 0],   # R
        centers[:, 1],   # G
        centers[:, 2],   # B
        c=colors,
        s=size_norm,
        alpha=0.85,
        edgecolors="none",
        depthshade=True,
    )

    ax3d.set_xlabel("Rosso (R)", color="white", labelpad=8)
    ax3d.set_ylabel("Verde (G)", color="white", labelpad=8)
    ax3d.set_zlabel("Blu (B)", color="white", labelpad=8)
    ax3d.set_title(title, color="white", pad=12, fontsize=13)
    ax3d.set_xlim(0, 255)
    ax3d.set_ylim(0, 255)
    ax3d.set_zlim(0, 255)

    # Stile assi scuro
    for pane in (ax3d.xaxis.pane, ax3d.yaxis.pane, ax3d.zaxis.pane):
        pane.fill = False
        pane.set_edgecolor("#333344")
    ax3d.tick_params(colors="gray")

    # ── Pannello destro: anteprima immagine ──────────────────────────────────
    ax_img = fig.add_subplot(122)
    ax_img.imshow(image)
    ax_img.set_title("Immagine originale", color="white", fontsize=13)
    ax_img.axis("off")

    # Testo informativo
    total_pixels = image.shape[0] * image.shape[1]
    unique_colors = len(counts)
    info = (
        f"Pixel totali: {total_pixels:,}\n"
        f"Colori unici (bins={bins}): {unique_colors}\n"
        f"Dimensione sfera ∝ frequenza"
    )
    fig.text(
        0.01, 0.02, info,
        color="gray", fontsize=8,
        va="bottom", family="monospace",
    )

    plt.tight_layout()
    plt.show()


# ─── Demo con immagine di test ────────────────────────────────────────────────

def generate_demo_image() -> np.ndarray:
    """Genera un'immagine demo colorata se non ne viene fornita una."""
    np.random.seed(42)
    h, w = 300, 300
    img = np.zeros((h, w, 3), dtype=np.uint8)

    # Gradiente di sfondo
    for i in range(h):
        img[i, :, 0] = int(i / h * 200)          # R crescente
        img[i, :, 2] = int((1 - i / h) * 200)    # B decrescente

    # Cerchi colorati sovrapposti
    for _ in range(12):
        cx, cy = np.random.randint(50, 250, 2)
        r = np.random.randint(20, 80)
        color = tuple(np.random.randint(50, 255, 3).tolist())
        cv2.circle(img, (cx, cy), r, color, -1)

    return img



## Frequenza colori immagine generata

In [ ]:

image = generate_demo_image()
title = "Frequenza dei colori – immagine demo"

plot_3d_color_frequency(
    image,
    bins=30,        # Aumenta per più dettaglio, diminuisci per aggregare di più
    top_n=None,     # Es. top_n=200 per mostrare solo i 200 colori più frequenti
    title=title,
)

## Frequenza colori ritaglio

In [ ]:

ruotata_tagliata = load_image('../Lezione2/out/Lanterna-300dpi_ruotata_tagliata.png')
title = f"Frequenza dei colori – ../Lezione2/out/Lanterna-300dpi_ruotata_tagliata.png"

plot_3d_color_frequency(
    ruotata_tagliata,
    bins=30,        # Aumenta per più dettaglio, diminuisci per aggregare di più
    top_n=None,     # Es. top_n=200 per mostrare solo i 200 colori più frequenti
    title=title,
)


## Frequenza colori ritaglio discretizzata

In [ ]:

sfondo = ruotata_tagliata[18:30,12:17,:]
title = f"Frequenza dei colori – sfondo"

plot_3d_color_frequency(
    sfondo,
    bins=30,        # Aumenta per più dettaglio, diminuisci per aggregare di più
    top_n=None,     # Es. top_n=200 per mostrare solo i 200 colori più frequenti
    title=title,
)


## Frequenza colori carta

In [ ]:

display(f'Dimensioni: {len(ruotata_tagliata)}')
sfondo = ruotata_tagliata[500:560,700:760,:]
title = f"Frequenza dei colori – sfondo"

plot_3d_color_frequency(
    sfondo,
    bins=30,        # Aumenta per più dettaglio, diminuisci per aggregare di più
    top_n=None,     # Es. top_n=200 per mostrare solo i 200 colori più frequenti
    title=title,
)


# Eliminare lo sfondo - suggerito da Gemini

In [ ]:
img_ruotata_tagliata = cv2.imread('../Lezione2/out/Lanterna-300dpi_ruotata_tagliata.png')

# 2. Conversione in scala di grigi
gray = cv2.cvtColor(img_ruotata_tagliata, cv2.COLOR_BGR2GRAY)

plt.imshow(gray, cmap='gray')
plt.title("Originale grigia")
plt.show()

# 3. Rimozione dello sfondo non uniforme (Morfologia Black Hat o Sottrazione)
# Creiamo un elemento strutturante abbastanza grande da coprire il testo ma non lo sfondo
kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (50, 50))
# La chiusura morfologica isola lo sfondo eliminando il testo scuro
background = cv2.morphologyEx(gray, cv2.MORPH_CLOSE, kernel)
# Sottraiamo l'immagine originale dallo sfondo per ottenere solo il testo su fondo bianco
corrected = cv2.divide(gray, background, scale=255)

# 4. Binarizzazione con il metodo di Otsu
# Questo calcola automaticamente la soglia ideale separando i due picchi dell'istogramma
_, thresh = cv2.threshold(corrected, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)

# 5. (Opzionale) Pulizia finale del rumore (pattern a righe residue)
# Un filtro mediano leggero rimuove i micro-pixel neri senza alterare i caratteri
final_img = cv2.medianBlur(thresh, 1)

plt.imshow(final_img, cmap='gray')
plt.title("Pulita")
plt.show()

# Esempi di utilizzo Pipeline

In [ ]:
from snippets.modules.pipeline import Pipeline
import PIL.Image

img_ruotata_tagliata = cv2.imread('../Lezione2/out/Lanterna-300dpi_ruotata_tagliata.png')

result = (
    Pipeline(img_ruotata_tagliata)
    .gaussian_blur(ksize=3)
    .run(to=PIL.Image.Image)
)

display(result)


In [ ]:
base = Pipeline(img_ruotata_tagliata).gaussian_blur(ksize=5).clahe(clip_limit=2.0)

varianti = [
    base.clone().update_step(1, clip_limit=1.0),
    base.clone().update_step(1, clip_limit=4.0),
    base.clone().update_step(1, clip_limit=8.0),
]

for i, v in enumerate(varianti):
    out = v.run(to="PIL")
    # print(f"  Variante {i}: mean brightness = {out.mean():.1f}")
    display(out)


In [ ]:
import ipywidgets as widgets
from IPython.display import display

value_input = widgets.FloatSlider(
    value=2.0,
    min=0.5,
    max=10.0,
    step=0.5,
    description='Clip Limit:')

value_input.observe(lambda change: image_out.update(
    Pipeline(img_ruotata_tagliata).gaussian_blur(ksize=5).clahe(clip_limit=change['new']).run(to="PIL")
), names='value')

display(value_input)

base = Pipeline(img_ruotata_tagliata).gaussian_blur(ksize=5).clahe(clip_limit=2.0)

out = base.run(to="PIL")
image_out = display(out, display_id=True)
